In [29]:
import json
import re
from pathlib import Path

import pandas as pd
import numpy as np
import pingouin as pg

import matplotlib.pyplot as plt
import seaborn as sns

from fau_colors import cmaps, register_fausans_font
import biopsykit as bp

from empkins_io.datasets.d03.macro_ap01 import MacroBaseDataset

%load_ext autoreload
%autoreload 2
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
register_fausans_font()
plt.close("all")

palette = sns.color_palette(cmaps.faculties)
sns.set_theme(context="notebook", style="ticks", palette=palette)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["mathtext.default"] = "regular"
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = "FAUSans Office"

palette

[(0.0, 0.1843137254901961, 0.4235294117647059),
 (0.4666666666666667, 0.6235294117647059, 0.7098039215686275),
 (1.0, 0.7215686274509804, 0.10980392156862745),
 (0.0, 0.6392156862745098, 0.8784313725490196),
 (0.2627450980392157, 0.6901960784313725, 0.16470588235294117),
 (0.7843137254901961, 0.06274509803921569, 0.1803921568627451)]

In [31]:
dataset = MacroBaseDataset(base_path=Path("/Volumes/luca_ssd/Study_Data/2022_05_AP01_Macro"), exclude_without_mocap=True)

In [32]:
df = bp.io.load_long_format_csv("/Users/abelluc/Code/empkins-d03-macro-analysis/experiments/2022_05_macro/00_general/feature_export/movement_features/movement_features_per_phase_for_classification.csv")

In [33]:
df = df.unstack(["condition", "phase", "feature_type", "body_part", "channel", "type", "metric", "axis"])
df = df.droplevel(0, axis=1)

In [34]:
df.columns = ["-".join(col) for col in df.columns]

In [35]:
df

,ftsst-talk-expert-Head-gyr-static_periods-count_per_min-norm,ftsst-talk-expert-Head-gyr-static_periods-max_duration_sec-norm,ftsst-talk-expert-Head-gyr-static_periods-mean_duration_sec-norm,ftsst-talk-expert-Head-gyr-static_periods-ratio_percent-norm,ftsst-talk-expert-Head-gyr-static_periods-std_duration_sec-norm,ftsst-talk-expert-Head-vel-static_periods-count_per_min-norm,ftsst-talk-expert-Head-vel-static_periods-max_duration_sec-norm,ftsst-talk-expert-Head-vel-static_periods-mean_duration_sec-norm,ftsst-talk-expert-Head-vel-static_periods-ratio_percent-norm,ftsst-talk-expert-Head-vel-static_periods-std_duration_sec-norm,...,tsst-talk-generic-jRightWrist-ang-fft_aggregated_variance-fft_aggregated_variance-norm,tsst-talk-generic-jRightWrist-ang-max_val-max_val-norm,tsst-talk-generic-jRightWrist-ang-mean-mean-norm,tsst-talk-generic-jRightWrist-ang-std-std-norm,tsst-talk-generic-jRightWrist-ang-zero_crossing-zero_crossing-x,tsst-talk-generic-jRightWrist-ang-zero_crossing-zero_crossing-y,tsst-talk-generic-jRightWrist-ang-zero_crossing-zero_crossing-z,tsst-talk-generic-jT1C7-ang-zero_crossing-zero_crossing-x,tsst-talk-generic-jT1C7-ang-zero_crossing-zero_crossing-y,tsst-talk-generic-jT1C7-ang-zero_crossing-zero_crossing-z
subject,,,,,,,,,,,,,,,,,,,,,
VP_01,11.399367,2.483,0.755368,14.351203,0.412668,14.999167,3.483,0.966707,24.166324,0.674523,...,5.338991e+05,63.960151,54.559057,6.485227,18.0,86.0,10.0,151.0,85.0,2.0
VP_02,21.398811,8.483,1.653822,58.983056,1.413562,15.199156,11.483,3.177303,80.487195,2.806263,...,9.037153e+05,77.700545,51.000098,8.475535,0.0,24.0,0.0,14.0,22.0,0.0
VP_04,16.038722,5.733,1.387605,37.092349,0.990996,18.810846,6.733,1.580684,49.556680,1.339437,...,1.011303e+06,43.277395,26.234381,3.571146,18.0,2.0,4.0,52.0,16.0,0.0
VP_05,20.162151,10.732,1.623686,54.561660,1.448920,24.002560,6.983,1.785256,71.417858,1.460168,...,8.087688e+05,31.789651,13.782244,4.156987,6.0,44.0,16.0,62.0,0.0,0.0
VP_06,11.919872,3.484,0.808367,16.059379,0.577263,22.647757,4.483,1.073202,40.509354,0.876260,...,1.760178e+06,49.757354,21.179977,12.518102,44.0,80.0,70.0,28.0,95.0,0.0
VP_07,18.896104,9.233,1.604392,50.527922,1.614908,13.636364,9.233,2.326071,52.865260,2.161059,...,1.118732e+06,62.220285,24.962655,5.932802,6.0,7.0,11.0,136.0,11.0,0.0
VP_08,9.835528,2.483,0.688260,11.282334,0.393426,25.769084,3.983,0.950901,40.839736,0.681650,...,9.454682e+05,66.693434,15.445233,3.442546,63.0,19.0,23.0,109.0,50.0,0.0
VP_09,14.949338,7.733,1.203347,29.982061,1.239020,18.935829,8.483,1.796411,56.694203,1.908417,...,1.118378e+06,75.823536,17.926837,4.526936,4.0,6.0,8.0,36.0,36.0,0.0
VP_10,19.538402,8.483,1.816576,59.154981,1.664792,19.341045,10.732,1.797020,57.927087,1.825181,...,2.337715e+06,46.524571,11.263491,6.929692,66.0,60.0,73.0,30.0,19.0,0.0


In [36]:
df.to_csv(dataset.data_tabular_path.joinpath("movement_features/final/movement_features_per_phase.csv"))